# SHL Assessment Recommendation System

## RAG Pipeline with Qwen/Qwen3-1.7B

**Pipeline:**
1. Web scraping of SHL product catalog (377+ Individual Test Solutions)
2. Sentence-transformer embeddings (`all-MiniLM-L6-v2`)
3. FAISS vector store for similarity search
4. Qwen/Qwen3-1.7B for query understanding & re-ranking
5. Mean Recall@K evaluation on labeled train set
6. Prediction generation for test set

In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import json
import re
import os
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)
CATALOG_JSON = os.path.join(DATA_DIR, 'shl_catalog.json')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU count: {torch.cuda.device_count()}')

PyTorch: 2.10.0+rocm7.1
CUDA available: True
GPU count: 8


## 1. Web Scraping SHL Product Catalog

Crawl the SHL product catalog to collect all **Individual Test Solutions** 
(ignoring Pre-packaged Job Solutions). For each assessment, scrape the detail page 
to collect: name, URL, description, test type, job levels, languages, and duration.

In [13]:
CATALOG_URL = 'https://www.shl.com/solutions/products/product-catalog/'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

TEST_TYPE_MAP = {
    'A': 'Ability & Aptitude',
    'B': 'Biodata & Situational Judgement',
    'C': 'Competencies',
    'D': 'Development & 360',
    'E': 'Assessment Exercises',
    'K': 'Knowledge & Skills',
    'P': 'Personality & Behavior',
    'S': 'Simulations',
}


def scrape_catalog_page(start=0):
    """Scrape one catalog page for Individual Test Solutions."""
    resp = requests.get(CATALOG_URL, params={'start': start}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    html = resp.text

    marker = 'Individual Test Solutions'
    idx = html.find(marker)
    if idx == -1:
        return []

    section_html = html[idx:]
    soup = BeautifulSoup(section_html, 'html.parser')

    assessments = []
    rows = soup.find_all('tr')
    for row in rows:
        link = row.find('a', href=True)
        if not link or '/product-catalog/view/' not in link['href']:
            continue
        name = link.get_text(strip=True)
        if not name or name == 'Back to Product Catalog':
            continue
        href = link['href']
        url = href if href.startswith('http') else f'https://www.shl.com{href}'

        tds = row.find_all('td')
        remote = False
        adaptive = False
        test_type_codes = ''
        if len(tds) >= 4:
            remote = bool(tds[1].find('span', class_=lambda c: c and 'catalogue' in c.lower()) if tds[1] else False)
            adaptive = bool(tds[2].find('span', class_=lambda c: c and 'catalogue' in c.lower()) if tds[2] else False)
            test_type_codes = tds[3].get_text(strip=True) if tds[3] else ''

        assessments.append({
            'name': name,
            'url': url,
            'remote_testing': remote,
            'adaptive_irt': adaptive,
            'test_type_codes': test_type_codes,
        })

    if not assessments:
        for link in soup.find_all('a', href=True):
            href = link['href']
            if '/product-catalog/view/' not in href:
                continue
            name = link.get_text(strip=True)
            if not name or name == 'Back to Product Catalog':
                continue
            url = href if href.startswith('http') else f'https://www.shl.com{href}'
            assessments.append({'name': name, 'url': url, 'remote_testing': False, 'adaptive_irt': False, 'test_type_codes': ''})

    return assessments


def scrape_detail_page(url):
    """Scrape metadata from an individual assessment detail page."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        text = soup.get_text(separator='\n')

        detail = {'url': url}

        h1 = soup.find('h1')
        detail['name'] = h1.get_text(strip=True) if h1 else ''

        h4s = soup.find_all('h4')
        for h4 in h4s:
            label = h4.get_text(strip=True).lower()
            sibling = h4.find_next_sibling()
            val = sibling.get_text(strip=True) if sibling else ''
            if 'description' in label:
                detail['description'] = val
            elif 'job level' in label:
                detail['job_levels'] = val
            elif 'language' in label:
                detail['languages'] = val

        detail.setdefault('description', '')
        detail.setdefault('job_levels', '')
        detail.setdefault('languages', '')

        dur = re.search(r'Completion Time in minutes\s*=\s*(\d+)', text)
        detail['duration_minutes'] = int(dur.group(1)) if dur else None

        tt = re.search(r'Test Type:\s*([A-Z ]+)', text)
        detail['test_type_codes'] = tt.group(1).strip() if tt else ''
        detail['test_type_names'] = ', '.join(
            TEST_TYPE_MAP.get(c, c) for c in detail['test_type_codes'].split() if c in TEST_TYPE_MAP
        )

        if 'Remote Testing' in text:
            after = text.split('Remote Testing')[1][:50]
            detail['remote_testing'] = 'yes' in after.lower()
        else:
            detail['remote_testing'] = False

        return detail

    except Exception as e:
        return {'url': url, 'name': '', 'description': '', 'error': str(e)}


print('Scraping functions defined.')

Scraping functions defined.


In [14]:
if os.path.exists(CATALOG_JSON):
    print(f'Loading cached catalog from {CATALOG_JSON}')
    with open(CATALOG_JSON) as f:
        catalog = json.load(f)
    print(f'Loaded {len(catalog)} assessments')
else:
    # Phase 1: Scrape catalog listing pages
    all_items = []
    seen_urls = set()
    for start in range(0, 500, 12):
        items = scrape_catalog_page(start)
        if not items:
            print(f'  start={start}: no items found, stopping.')
            break
        new = [x for x in items if x['url'] not in seen_urls]
        seen_urls.update(x['url'] for x in new)
        all_items.extend(new)
        print(f'  start={start}: {len(items)} found, {len(new)} new, total={len(all_items)}')
        time.sleep(1.5)

    print(f'\nTotal unique assessments from catalog: {len(all_items)}')

    # Phase 2: Scrape detail pages
    catalog = []
    for i, item in enumerate(all_items):
        if (i + 1) % 20 == 0 or i == 0:
            print(f'  Scraping detail page [{i+1}/{len(all_items)}]...')
        detail = scrape_detail_page(item['url'])
        if not detail.get('test_type_codes') and item.get('test_type_codes'):
            detail['test_type_codes'] = item['test_type_codes']
            detail['test_type_names'] = ', '.join(
                TEST_TYPE_MAP.get(c, c) for c in detail['test_type_codes'].split() if c in TEST_TYPE_MAP
            )
        if not detail.get('remote_testing'):
            detail['remote_testing'] = item.get('remote_testing', False)
        if not detail.get('adaptive_irt'):
            detail['adaptive_irt'] = item.get('adaptive_irt', False)
        catalog.append(detail)
        time.sleep(1)

    with open(CATALOG_JSON, 'w') as f:
        json.dump(catalog, f, indent=2)
    print(f'\nSaved {len(catalog)} assessments to {CATALOG_JSON}')

df = pd.DataFrame(catalog)
if 'error' in df.columns:
    err_count = df['error'].notna().sum()
    if err_count > 0:
        print(f'\nWarning: {err_count} assessments had scraping errors')
        df = df[df['error'].isna()].copy()
print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

Loading cached catalog from data/shl_catalog.json
Loaded 377 assessments

Dataset shape: (377, 10)
Columns: ['url', 'name', 'description', 'job_levels', 'languages', 'duration_minutes', 'test_type_codes', 'test_type_names', 'remote_testing', 'adaptive_irt']


,url,name,description,job_levels,languages,duration_minutes,test_type_codes,test_type_names,remote_testing,adaptive_irt
0,https://www.shl.com/products/product-catalog/v...,Global Skills Development Report,This report is designed to be given to individ...,"Director, Entry-Level, Executive, General Popu...",,NaN,A,Ability & Aptitude,True,False
1,https://www.shl.com/products/product-catalog/v...,.NET Framework 4.5,The.NET Framework 4.5 test measures knowledge ...,"Professional Individual Contributor, Mid-Profe...","English (USA),",30.0,K,Knowledge & Skills,True,True
2,https://www.shl.com/products/product-catalog/v...,.NET MVC (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",17.0,K,Knowledge & Skills,True,False
3,https://www.shl.com/products/product-catalog/v...,.NET MVVM (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",5.0,K,Knowledge & Skills,True,False
4,https://www.shl.com/products/product-catalog/v...,.NET WCF (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",11.0,K,Knowledge & Skills,True,False
5,https://www.shl.com/products/product-catalog/v...,.NET WPF (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",9.0,K,Knowledge & Skills,True,False
6,https://www.shl.com/products/product-catalog/v...,.NET XAML (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",5.0,K,Knowledge & Skills,True,False
7,https://www.shl.com/products/product-catalog/v...,Accounts Payable (New),Multiple-choice test that measures the knowled...,"Entry-Level, Graduate, Mid-Professional, Profe...","English (USA),",9.0,K,Knowledge & Skills,True,False
8,https://www.shl.com/products/product-catalog/v...,Accounts Payable Simulation (New),Simulated data entry test that measures the ab...,"Entry-Level, Graduate, Mid-Professional, Profe...","English (USA),",8.0,S,Simulations,True,False
9,https://www.shl.com/products/product-catalog/v...,Accounts Receivable (New),Multiple-choice test that measures the knowled...,"Entry-Level, Graduate, Mid-Professional, Profe...","English (USA),",13.0,K,Knowledge & Skills,True,False


## 2. Data Processing, Embeddings & FAISS Index

Create rich text representations for each assessment, generate embeddings with 
`all-MiniLM-L6-v2`, and build a FAISS index for cosine similarity search.

In [15]:
def create_text_repr(row):
    """Build a rich text string combining all assessment metadata for embedding."""
    parts = [f"Assessment: {row.get('name', '')}"]
    if row.get('description'):
        parts.append(f"Description: {row['description']}")
    if row.get('test_type_names'):
        parts.append(f"Test Type: {row['test_type_names']}")
    elif row.get('test_type_codes'):
        codes = row['test_type_codes']
        names = ', '.join(TEST_TYPE_MAP.get(c, c) for c in codes.split() if c in TEST_TYPE_MAP)
        parts.append(f"Test Type: {names}")
    if row.get('job_levels'):
        parts.append(f"Job Levels: {row['job_levels']}")
    if row.get('duration_minutes'):
        parts.append(f"Duration: {row['duration_minutes']} minutes")
    if row.get('languages'):
        parts.append(f"Languages: {row['languages']}")
    return '. '.join(parts)


df['text_repr'] = df.apply(create_text_repr, axis=1)

print('Sample text representation:')
print(df['text_repr'].iloc[0])
print()

# Generate embeddings
EMBED_MODEL_NAME = 'all-MiniLM-L6-v2'
print(f'Loading embedding model: {EMBED_MODEL_NAME}')
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print('Generating embeddings for all assessments...')
embeddings = embed_model.encode(
    df['text_repr'].tolist(),
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)
embeddings = np.array(embeddings, dtype='float32')

# Build FAISS index (inner product on normalized vectors = cosine similarity)
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings)

print(f'\nFAISS index built: {faiss_index.ntotal} vectors, dim={dim}')

Sample text representation:
Assessment: Global Skills Development Report. Description: This report is designed to be given to individuals who have completed the Global Skills Assessment (GSA). With coverage across the Great 8 Domains, this measure of self-reported behaviors offers a complete overview of their current skills. Participants receive actionable tips on leveraging their top skill strengths and how they might develop their growth skills.. Test Type: Ability & Aptitude. Job Levels: Director, Entry-Level, Executive, General Population, Graduate, Manager, Mid-Professional, Front Line Manager, Supervisor,. Duration: nan minutes

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2765.01it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for all assessments...


Batches: 100%|██████████| 6/6 [00:00<00:00, 56.64it/s]


FAISS index built: 377 vectors, dim=384


## 3. Load Qwen/Qwen3-1.7B & Build RAG Pipeline

The RAG pipeline:
1. Embeds the query and retrieves top candidates via FAISS
2. Uses Qwen to re-rank candidates based on relevance to the full query
3. Balances recommendations across test types (technical + behavioral)

In [16]:
QWEN_MODEL = 'Qwen/Qwen3-1.7B'
print(f'Loading {QWEN_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL, trust_remote_code=True)
qwen = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True
)
qwen.eval()
print('Model loaded.\n')


def generate_response(prompt, max_tokens=256, temperature=0.3):
    """Generate text from the Qwen model."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(qwen.device)
    with torch.no_grad():
        out = qwen.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


def retrieve_candidates(query, k=30):
    """Retrieve top-k candidates from FAISS."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = faiss_index.search(q_emb, k)
    cands = df.iloc[indices[0]].copy()
    cands['sim_score'] = scores[0]
    return cands.reset_index(drop=True)


def rerank_with_llm(query, candidates, top_k=10):
    """Use Qwen to re-rank retrieved candidates."""
    context_lines = []
    for i, (_, row) in enumerate(candidates.iterrows()):
        line = f"{i+1}. {row['name']}"
        if row.get('description'):
            line += f" - {str(row['description'])[:120]}"
        tc = row.get('test_type_codes', '')
        if tc:
            line += f" [Type: {tc}]"
        dur = row.get('duration_minutes')
        if dur:
            line += f" [{dur} min]"
        context_lines.append(line)
    context = '\n'.join(context_lines)

    prompt = (
        f"Given this hiring query, select the {top_k} most relevant assessments "
        f"from the list below. Consider BOTH technical skills AND soft/behavioral "
        f"skills mentioned. Also respect any duration or level constraints.\n\n"
        f"Query: {query}\n\n"
        f"Assessments:\n{context}\n\n"
        f"Return ONLY the numbers of the {top_k} most relevant assessments, "
        f"comma-separated, most relevant first. Example: 3,1,7,5,2,8,4,6,9,10\n\n"
        f"Selected:"
    )

    response = generate_response(prompt, max_tokens=100, temperature=0.1)

    numbers = re.findall(r'\d+', response)
    selected = []
    for n in numbers:
        idx = int(n) - 1
        if 0 <= idx < len(candidates) and idx not in selected:
            selected.append(idx)
        if len(selected) >= top_k:
            break

    # Fallback: fill remaining with highest similarity scores
    for i in range(len(candidates)):
        if len(selected) >= top_k:
            break
        if i not in selected:
            selected.append(i)

    return candidates.iloc[selected[:top_k]]


def recommend(query, top_k=5, retrieve_k=30):
    """Full RAG recommendation pipeline."""
    candidates = retrieve_candidates(query, k=retrieve_k)
    results = rerank_with_llm(query, candidates, top_k=top_k)
    cols = ['name', 'url', 'test_type_codes', 'duration_minutes', 'sim_score']
    cols = [c for c in cols if c in results.columns]
    return results[cols].reset_index(drop=True)


# Quick test
test_q = 'I am hiring for Machine Learning developers can develop and deploy GenAI models.'
print(f'Test Query: {test_q}\n')
test_results = recommend(test_q)
for i, (_, row) in enumerate(test_results.iterrows()):
    print(f"{i+1}. {row['name']} [{row.get('test_type_codes','')}] - {row['url']}")

Loading Qwen/Qwen3-1.7B...


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1786.30it/s, Materializing param=model.norm.weight]                             
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded.

Test Query: I am hiring for Machine Learning developers can develop and deploy GenAI models.

1. Geoinformatics Engineering (New) [K] - https://www.shl.com/products/product-catalog/view/geoinformatics-engineering-new/
2. Automata Data Science Pro (New) [S] - https://www.shl.com/products/product-catalog/view/automata-data-science-pro-new/
3. Automata Pro (New) [S] - https://www.shl.com/products/product-catalog/view/automata-pro-new/
4. Automata (New) [S] - https://www.shl.com/products/product-catalog/view/automata-new/
5. Virtual Assessment and Development Centers [P] - https://www.shl.com/products/product-catalog/view/virtual-assessment-and-development-centers/


## 4. Evaluation on Train Set

Load the labeled train data (10 queries with human-labeled relevant assessments) and compute **Mean Recall@10**.

$$\text{Recall@K} = \frac{\text{Number of relevant assessments in top K}}{\text{Total relevant assessments for the query}}$$

$$\text{Mean Recall@K} = \frac{1}{N} \sum_{i=1}^{N} \text{Recall@K}_i$$

In [17]:
train_df = pd.read_excel('Gen_AI Dataset.xlsx', sheet_name='Train-Set')
test_df = pd.read_excel('Gen_AI Dataset.xlsx', sheet_name='Test-Set')

print(f'Train: {train_df.shape[0]} rows, {train_df["Query"].nunique()} unique queries')
print(f'Test:  {test_df.shape[0]} queries')

train_grouped = train_df.groupby('Query')['Assessment_url'].apply(list).reset_index()


def normalize_url(url):
    """Normalize SHL URLs for fair comparison."""
    url = url.strip().rstrip('/')
    url = url.replace('https://www.shl.com/solutions/products/', 'https://www.shl.com/products/')
    return url.lower()


def recall_at_k(predicted_urls, actual_urls, k=10):
    pred = {normalize_url(u) for u in predicted_urls[:k]}
    actual = {normalize_url(u) for u in actual_urls}
    if not actual:
        return 0.0
    return len(pred & actual) / len(actual)


print('\n--- Evaluation on Train Set ---\n')
recalls = []
for _, row in train_grouped.iterrows():
    query = row['Query']
    actual_urls = row['Assessment_url']

    results = recommend(query, top_k=10)
    predicted_urls = results['url'].tolist()

    r = recall_at_k(predicted_urls, actual_urls)
    recalls.append(r)
    print(f'Query: {query[:80]}...')
    print(f'  Relevant: {len(actual_urls)} | Recall@10: {r:.3f}')
    print()

mean_recall = np.mean(recalls)
print(f'=========================================')
print(f'  Mean Recall@10: {mean_recall:.4f}')
print(f'=========================================')

Train: 65 rows, 10 unique queries
Test:  9 queries

--- Evaluation on Train Set ---



Query: Based on the JD below recommend me assessment for the Consultant position in my ...
  Relevant: 5 | Recall@10: 0.000

Query: Content Writer required, expert in English and SEO....
  Relevant: 5 | Recall@10: 0.400

Query: Find me 1 hour long assesment for the below job at SHL
Job Description

 Join a ...
  Relevant: 9 | Recall@10: 0.000

Query: I am hiring for Java developers who can also collaborate effectively with my bus...
  Relevant: 5 | Recall@10: 0.600

Query: I am looking for a COO for my company in China and I want to see if they are cul...
  Relevant: 6 | Recall@10: 0.167

Query: I want to hire a Senior Data Analyst with 5 years of experience and expertise in...
  Relevant: 10 | Recall@10: 0.200

Query: I want to hire new graduates for a sales role in my company, the budget is for a...
  Relevant: 9 | Recall@10: 0.111

Query: ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 ...
  Relevant: 6 | Recall@10: 0.167

Query: KEY RESPONSIBITILES:


## 6. Pipeline Improvements

Issues identified:
1. **11 train URLs are Pre-packaged Job Solutions** not in our scraped catalog → recall ceiling
2. **Shallow retrieval** (top-30) misses relevant items for complex queries
3. **Embedding model** `all-MiniLM-L6-v2` is lightweight but not optimal for retrieval
4. **URL normalization** needs slug-based comparison for robustness

**Fixes applied below:**
- Scrape Pre-packaged solutions and add to catalog
- Switch to `all-mpnet-base-v2` (higher quality embeddings)
- Increase retrieval pool to top-100
- Multi-query retrieval: decompose query into aspects, search each, merge results
- Slug-based URL comparison for evaluation

In [19]:
# --- Step 1: Scrape Pre-packaged Job Solutions and add to catalog ---

PREPACKAGED_JSON = os.path.join(DATA_DIR, 'shl_prepackaged.json')

def scrape_prepackaged_page(start=0):
    """Scrape one catalog page for Pre-packaged Job Solutions."""
    resp = requests.get(CATALOG_URL, params={'start': start}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    html = resp.text

    pre_marker = 'Pre-packaged Job Solutions'
    ind_marker = 'Individual Test Solutions'
    pre_idx = html.find(pre_marker)
    ind_idx = html.find(ind_marker)
    if pre_idx == -1:
        return []

    section_html = html[pre_idx:ind_idx] if ind_idx > pre_idx else html[pre_idx:]
    soup = BeautifulSoup(section_html, 'html.parser')

    items = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if '/product-catalog/view/' not in href:
            continue
        name = link.get_text(strip=True)
        if not name or name == 'Back to Product Catalog':
            continue
        url = href if href.startswith('http') else f'https://www.shl.com{href}'
        items.append({'name': name, 'url': url})
    return items


if os.path.exists(PREPACKAGED_JSON):
    print(f'Loading cached pre-packaged from {PREPACKAGED_JSON}')
    with open(PREPACKAGED_JSON) as f:
        prepackaged = json.load(f)
else:
    all_pre = []
    seen = set()
    for start in range(0, 500, 12):
        items = scrape_prepackaged_page(start)
        if not items:
            print(f'  start={start}: no pre-packaged found, stopping.')
            break
        new = [x for x in items if x['url'] not in seen]
        seen.update(x['url'] for x in new)
        all_pre.extend(new)
        print(f'  start={start}: {len(items)} found, {len(new)} new, total={len(all_pre)}')
        time.sleep(1.5)

    print(f'\nTotal Pre-packaged: {len(all_pre)}')

    prepackaged = []
    for i, item in enumerate(all_pre):
        if (i + 1) % 20 == 0 or i == 0:
            print(f'  Detail [{i+1}/{len(all_pre)}]...')
        detail = scrape_detail_page(item['url'])
        detail['category'] = 'Pre-packaged'
        prepackaged.append(detail)
        time.sleep(1)

    with open(PREPACKAGED_JSON, 'w') as f:
        json.dump(prepackaged, f, indent=2)
    print(f'Saved {len(prepackaged)} pre-packaged to {PREPACKAGED_JSON}')

# Mark existing catalog items
for item in catalog:
    item.setdefault('category', 'Individual')

# Merge
full_catalog = catalog + prepackaged
df_full = pd.DataFrame(full_catalog)
if 'error' in df_full.columns:
    df_full = df_full[df_full['error'].isna()].copy()

print(f'\nFull catalog: {len(df_full)} assessments ({len(catalog)} Individual + {len(prepackaged)} Pre-packaged)')
df_full.head()

  start=0: 12 found, 12 new, total=12
  start=12: 12 found, 12 new, total=24
  start=24: 12 found, 12 new, total=36
  start=36: 12 found, 12 new, total=48
  start=48: 12 found, 12 new, total=60
  start=60: 12 found, 12 new, total=72
  start=72: 12 found, 12 new, total=84
  start=84: 12 found, 12 new, total=96
  start=96: 12 found, 12 new, total=108
  start=108: 12 found, 12 new, total=120
  start=120: 12 found, 12 new, total=132
  start=132: 9 found, 9 new, total=141
  start=144: no pre-packaged found, stopping.

Total Pre-packaged: 141
  Detail [1/141]...
  Detail [20/141]...
  Detail [40/141]...
  Detail [60/141]...
  Detail [80/141]...
  Detail [100/141]...
  Detail [120/141]...
  Detail [140/141]...
Saved 141 pre-packaged to data/shl_prepackaged.json

Full catalog: 518 assessments (377 Individual + 141 Pre-packaged)


,url,name,description,job_levels,languages,duration_minutes,test_type_codes,test_type_names,remote_testing,adaptive_irt,category
0,https://www.shl.com/products/product-catalog/v...,Global Skills Development Report,This report is designed to be given to individ...,"Director, Entry-Level, Executive, General Popu...",,NaN,A,Ability & Aptitude,True,False,Individual
1,https://www.shl.com/products/product-catalog/v...,.NET Framework 4.5,The.NET Framework 4.5 test measures knowledge ...,"Professional Individual Contributor, Mid-Profe...","English (USA),",30.0,K,Knowledge & Skills,True,True,Individual
2,https://www.shl.com/products/product-catalog/v...,.NET MVC (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",17.0,K,Knowledge & Skills,True,False,Individual
3,https://www.shl.com/products/product-catalog/v...,.NET MVVM (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",5.0,K,Knowledge & Skills,True,False,Individual
4,https://www.shl.com/products/product-catalog/v...,.NET WCF (New),Multi-choice test that measures the knowledge ...,"Mid-Professional, Professional Individual Cont...","English (USA),",11.0,K,Knowledge & Skills,True,False,Individual


In [20]:
# --- Step 2: Better embeddings + FAISS on full catalog ---

df_full['text_repr'] = df_full.apply(create_text_repr, axis=1)

EMBED_MODEL_V2 = 'all-mpnet-base-v2'
print(f'Loading improved embedding model: {EMBED_MODEL_V2}')
embed_model_v2 = SentenceTransformer(EMBED_MODEL_V2)

print('Generating embeddings for full catalog...')
embeddings_v2 = embed_model_v2.encode(
    df_full['text_repr'].tolist(),
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)
embeddings_v2 = np.array(embeddings_v2, dtype='float32')

dim_v2 = embeddings_v2.shape[1]
faiss_index_v2 = faiss.IndexFlatIP(dim_v2)
faiss_index_v2.add(embeddings_v2)

print(f'\nImproved FAISS index: {faiss_index_v2.ntotal} vectors, dim={dim_v2}')

Loading improved embedding model: all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2465.62it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for full catalog...


Batches: 100%|██████████| 9/9 [00:00<00:00, 27.34it/s]


Improved FAISS index: 518 vectors, dim=768


In [21]:
# --- Step 3: Improved RAG pipeline ---

def extract_query_aspects(query):
    """Use Qwen to decompose a complex query into searchable aspects."""
    if len(query) < 150:
        return [query]

    prompt = (
        "Extract the key hiring requirements from this query as short search phrases. "
        "Return one phrase per line, covering: job role, technical skills, soft skills, "
        "and any constraints (duration, level). Maximum 5 phrases.\n\n"
        f"Query: {query[:1500]}\n\nKey phrases:"
    )
    response = generate_response(prompt, max_tokens=150, temperature=0.1)
    lines = [l.strip().lstrip('0123456789.-) ') for l in response.split('\n') if l.strip()]
    aspects = [l for l in lines if len(l) > 5 and len(l) < 200]
    if not aspects:
        return [query[:500]]
    return aspects[:5]


def retrieve_multi_query(query, k=100):
    """Multi-aspect retrieval: decompose query, search each aspect, merge results."""
    aspects = extract_query_aspects(query)
    all_queries = [query[:500]] + aspects

    candidate_scores = {}
    for q in all_queries:
        q_emb = embed_model_v2.encode([q], normalize_embeddings=True).astype('float32')
        scores, indices = faiss_index_v2.search(q_emb, min(k, faiss_index_v2.ntotal))
        for score, idx in zip(scores[0], indices[0]):
            if idx in candidate_scores:
                candidate_scores[idx] = max(candidate_scores[idx], float(score))
            else:
                candidate_scores[idx] = float(score)

    sorted_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in sorted_candidates[:k]]
    top_scores = [score for _, score in sorted_candidates[:k]]

    result = df_full.iloc[top_indices].copy()
    result['sim_score'] = top_scores
    return result.reset_index(drop=True)


def recommend_v2(query, top_k=10, retrieve_k=100):
    """Improved recommendation pipeline with multi-query retrieval + LLM re-ranking."""
    candidates = retrieve_multi_query(query, k=retrieve_k)

    rerank_pool = candidates.head(30)
    reranked = rerank_with_llm(query, rerank_pool, top_k=top_k)
    cols = [c for c in ['name', 'url', 'test_type_codes', 'duration_minutes', 'sim_score'] if c in reranked.columns]
    return reranked[cols].reset_index(drop=True)


# Quick test
test_q = 'I am hiring for Java developers who can also collaborate effectively with my business teams.'
print(f'Query: {test_q}\n')
res = recommend_v2(test_q)
for i, (_, row) in enumerate(res.iterrows()):
    print(f"{i+1}. {row['name']} [{row.get('test_type_codes','')}] - {row['url']}")

Query: I am hiring for Java developers who can also collaborate effectively with my business teams.

1. Enterprise Java Beans (New) [K] - https://www.shl.com/products/product-catalog/view/enterprise-java-beans-new/
2. Java 2 Platform Enterprise Edition 1.4 Fundamental [K] - https://www.shl.com/products/product-catalog/view/java-2-platform-enterprise-edition-1-4-fundamental/
3. Manager 7.1 (International) [B] - https://www.shl.com/products/product-catalog/view/manager-7-1-solution/
4. Java Platform Enterprise Edition 7 (Java EE 7) [K] - https://www.shl.com/products/product-catalog/view/java-platform-enterprise-edition-7-java-ee-7/
5. Java Frameworks (New) [K] - https://www.shl.com/products/product-catalog/view/java-frameworks-new/
6. Java Web Services (New) [K] - https://www.shl.com/products/product-catalog/view/java-web-services-new/
7. Core Java (Entry Level) (New) [K] - https://www.shl.com/products/product-catalog/view/core-java-entry-level-new/
8. Core Java (Advanced Level) (New) [K

In [22]:
# --- Step 4: Re-evaluate with improved pipeline ---

def url_to_slug(url):
    """Extract URL slug for robust comparison."""
    return url.strip().rstrip('/').split('/')[-1].lower()


def recall_at_k_v2(predicted_urls, actual_urls, k=10):
    """Recall@K using slug-based URL comparison."""
    pred_slugs = {url_to_slug(u) for u in predicted_urls[:k]}
    actual_slugs = {url_to_slug(u) for u in actual_urls}
    if not actual_slugs:
        return 0.0
    return len(pred_slugs & actual_slugs) / len(actual_slugs)


print('--- Improved Evaluation on Train Set ---\n')
recalls_v2 = []
for _, row in train_grouped.iterrows():
    query = row['Query']
    actual_urls = row['Assessment_url']

    results = recommend_v2(query, top_k=10)
    predicted_urls = results['url'].tolist()

    r = recall_at_k_v2(predicted_urls, actual_urls)
    recalls_v2.append(r)
    print(f'Query: {query[:80]}...')
    print(f'  Relevant: {len(actual_urls)} | Recall@10: {r:.3f}')
    print()

mean_recall_v2 = np.mean(recalls_v2)
print(f'=========================================')
print(f'  Previous Mean Recall@10: {mean_recall:.4f}')
print(f'  Improved Mean Recall@10: {mean_recall_v2:.4f}')
print(f'  Improvement: +{mean_recall_v2 - mean_recall:.4f}')
print(f'=========================================')

--- Improved Evaluation on Train Set ---

Query: Based on the JD below recommend me assessment for the Consultant position in my ...
  Relevant: 5 | Recall@10: 0.000

Query: Content Writer required, expert in English and SEO....
  Relevant: 5 | Recall@10: 0.400

Query: Find me 1 hour long assesment for the below job at SHL
Job Description

 Join a ...
  Relevant: 9 | Recall@10: 0.000

Query: I am hiring for Java developers who can also collaborate effectively with my bus...
  Relevant: 5 | Recall@10: 0.600

Query: I am looking for a COO for my company in China and I want to see if they are cul...
  Relevant: 6 | Recall@10: 0.167

Query: I want to hire a Senior Data Analyst with 5 years of experience and expertise in...
  Relevant: 10 | Recall@10: 0.100

Query: I want to hire new graduates for a sales role in my company, the budget is for a...
  Relevant: 9 | Recall@10: 0.333

Query: ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 ...
  Relevant: 6 | Reca

In [23]:
# --- Step 5: Re-generate predictions on test set ---

predictions_v2 = []

for i, row in test_df.iterrows():
    query = row['Query']
    print(f'[{i+1}/{len(test_df)}] Query: {query[:80]}...')

    results = recommend_v2(query, top_k=10)
    for _, r in results.iterrows():
        predictions_v2.append({'Query': query, 'Assessment_url': r['url']})
    print(f'  -> {len(results)} recommendations\n')

pred_v2_df = pd.DataFrame(predictions_v2)
pred_v2_df.to_csv('predictions.csv', index=False)
print(f'Updated predictions saved to predictions.csv ({len(pred_v2_df)} rows)')
pred_v2_df.head(20)

[1/9] Query: Looking to hire mid-level professionals who are proficient in Python, SQL and Ja...
  -> 10 recommendations

[2/9] Query: Job Description

 Join a community that is shaping the future of work! 

 SHL, P...
  -> 10 recommendations

[3/9] Query: I am hiring for an analyst and wants applications to screen using Cognitive and ...
  -> 10 recommendations

[4/9] Query: I have a JD Job Description

 People Science. People Answers  !

Do you love con...
  -> 10 recommendations

[5/9] Query: I am new looking for new graduates in my sales team, suggest an 30 min long asse...
  -> 10 recommendations

[6/9] Query: For Marketing - Content Writer Position

Department: Marketing

Location: Gurugr...
  -> 10 recommendations

[7/9] Query: I want to hire a product manager with 3-4 years of work experience and expertise...
  -> 10 recommendations

[8/9] Query: Suggest me an assessment for the JD below Job Description

 Find purpose in each...
  -> 10 recommendations

[9/9] Query: I want to h

,Query,Assessment_url
0,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
1,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
2,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
3,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
4,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
5,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
6,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
7,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
8,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
9,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...


## 7. Diagnosis & Further Improvements

Let's diagnose the 0.000 recall queries, then implement:
1. **Hybrid search** (semantic + keyword overlap) to catch exact skill matches
2. **No LLM re-ranking** — test if Qwen re-ranking is actually hurting recall
3. **Better query compression** for long JD texts

In [24]:
# --- Diagnose 0.000 recall queries ---

for _, row in train_grouped.iterrows():
    query = row['Query']
    actual_urls = row['Assessment_url']
    results = recommend_v2(query, top_k=10)
    predicted_urls = results['url'].tolist()

    r = recall_at_k_v2(predicted_urls, actual_urls)
    if r < 0.01:
        print(f'=== ZERO RECALL QUERY ===')
        print(f'Query: {query[:120]}...\n')
        print('Expected (ground truth):')
        for u in actual_urls:
            slug = url_to_slug(u)
            in_catalog = slug in {url_to_slug(x) for x in df_full['url']}
            print(f'  {"[OK]" if in_catalog else "[MISSING]"} {slug}')
        print('\nPredicted:')
        for _, r_row in results.iterrows():
            print(f'  {url_to_slug(r_row["url"])} ({r_row.get("sim_score", 0):.3f})')
        print('\n' + '-'*60 + '\n')

=== ZERO RECALL QUERY ===
Query: Based on the JD below recommend me assessment for the Consultant position in my organizations. The assessment should not...

Expected (ground truth):
  [OK] shl-verify-interactive-numerical-calculation
  [OK] administrative-professional-short-form
  [OK] verify-verbal-ability-next-generation
  [OK] occupational-personality-questionnaire-opq32r
  [OK] professional-7-1-solution

Predicted:
  manager-7-0-solution (0.554)
  professionalindividual-contributor-short-form (0.674)
  retail-consultant-solution (0.600)
  assessment-and-development-center-exercises (0.599)
  verify-following-instructions (0.583)
  verify-interactive-ability-report (0.572)
  customer-service-short-form (0.567)
  manager-short-form (0.567)
  supervisor-short-form (0.566)
  graduate-7-1-job-focused-assessment (0.554)

------------------------------------------------------------

=== ZERO RECALL QUERY ===
Query: Find me 1 hour long assesment for the below job at SHL
Job Description

 

In [25]:
# --- Hybrid Search: Semantic + Keyword scoring ---

from collections import Counter
import string

def tokenize(text):
    """Simple whitespace + punctuation tokenizer, lowercased."""
    text = text.lower()
    text = text.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))
    return [w for w in text.split() if len(w) > 2]


def keyword_scores(query_tokens, corpus_tokens_list):
    """Compute Jaccard-like keyword overlap scores."""
    query_set = set(query_tokens)
    scores = []
    for doc_tokens in corpus_tokens_list:
        doc_set = set(doc_tokens)
        if not query_set or not doc_set:
            scores.append(0.0)
            continue
        overlap = len(query_set & doc_set)
        scores.append(overlap / len(query_set))
    return np.array(scores, dtype='float32')


# Pre-tokenize all assessment text representations
corpus_tokens = [tokenize(t) for t in df_full['text_repr'].tolist()]
print(f'Tokenized {len(corpus_tokens)} assessments')


def hybrid_retrieve(query, k=50, alpha=0.6):
    """Hybrid retrieval: alpha * semantic + (1-alpha) * keyword."""
    # Semantic scores
    q_emb = embed_model_v2.encode([query[:500]], normalize_embeddings=True).astype('float32')
    sem_scores_all, sem_indices = faiss_index_v2.search(q_emb, faiss_index_v2.ntotal)
    sem_scores = np.zeros(len(df_full), dtype='float32')
    for score, idx in zip(sem_scores_all[0], sem_indices[0]):
        sem_scores[idx] = score

    # Keyword scores
    q_tokens = tokenize(query)
    kw_scores = keyword_scores(q_tokens, corpus_tokens)

    # Normalize both to [0, 1]
    if sem_scores.max() > 0:
        sem_scores = sem_scores / sem_scores.max()
    if kw_scores.max() > 0:
        kw_scores = kw_scores / kw_scores.max()

    # Combine
    combined = alpha * sem_scores + (1 - alpha) * kw_scores

    top_indices = np.argsort(combined)[::-1][:k]
    result = df_full.iloc[top_indices].copy()
    result['sim_score'] = combined[top_indices]
    result['sem_score'] = sem_scores[top_indices]
    result['kw_score'] = kw_scores[top_indices]
    return result.reset_index(drop=True)


def recommend_v3(query, top_k=10):
    """Hybrid retrieval WITHOUT LLM re-ranking."""
    candidates = hybrid_retrieve(query, k=top_k)
    cols = [c for c in ['name', 'url', 'test_type_codes', 'duration_minutes', 'sim_score'] if c in candidates.columns]
    return candidates[cols].reset_index(drop=True)


# Quick test
test_q = 'I am hiring for Java developers who can also collaborate effectively with my business teams.'
print(f'\nQuery: {test_q}\n')
res = recommend_v3(test_q)
for i, (_, row) in enumerate(res.iterrows()):
    print(f"{i+1}. {row['name']} [{row.get('test_type_codes','')}]")

Tokenized 518 assessments

Query: I am hiring for Java developers who can also collaborate effectively with my business teams.

1. Java 2 Platform Enterprise Edition 1.4 Fundamental [K]
2. General Entry Level - All Industries 7.1(Americas) [B]
3. PJM Selection Report [A]
4. General Entry Level - All Industries 7.1 Solution [B]
5. General Entry Level - All Industries 7.0 Solution [B]
6. Java Platform Enterprise Edition 7 (Java EE 7) [K]
7. Business Communications [K]
8. Contact Center Team Lead/Coach - Short Form [A]
9. PJM Development Report [C]
10. Retail Manager w/ Sales Solution [A]


In [26]:
# --- Sweep alpha values and test with/without re-ranking ---

best_alpha = 0.5
best_recall = 0.0

print('Sweeping alpha (semantic weight) values...\n')
for alpha in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    recalls = []
    for _, row in train_grouped.iterrows():
        query = row['Query']
        actual_urls = row['Assessment_url']
        candidates = hybrid_retrieve(query, k=10, alpha=alpha)
        predicted_urls = candidates['url'].tolist()
        r = recall_at_k_v2(predicted_urls, actual_urls)
        recalls.append(r)
    mr = np.mean(recalls)
    print(f'  alpha={alpha:.1f} -> Mean Recall@10: {mr:.4f}')
    if mr > best_recall:
        best_recall = mr
        best_alpha = alpha

print(f'\nBest alpha: {best_alpha} with Mean Recall@10: {best_recall:.4f}')
print(f'(Previous best: {mean_recall_v2:.4f})')

Sweeping alpha (semantic weight) values...

  alpha=0.3 -> Mean Recall@10: 0.0844
  alpha=0.4 -> Mean Recall@10: 0.1044
  alpha=0.5 -> Mean Recall@10: 0.1156
  alpha=0.6 -> Mean Recall@10: 0.1667
  alpha=0.7 -> Mean Recall@10: 0.2167
  alpha=0.8 -> Mean Recall@10: 0.2000
  alpha=0.9 -> Mean Recall@10: 0.1967
  alpha=1.0 -> Mean Recall@10: 0.2133

Best alpha: 0.7 with Mean Recall@10: 0.2167
(Previous best: 0.2133)


In [27]:
# --- Final evaluation with best alpha ---

def recommend_final(query, top_k=10):
    """Final pipeline: hybrid retrieval with tuned alpha, no LLM re-ranking."""
    candidates = hybrid_retrieve(query, k=top_k, alpha=best_alpha)
    cols = [c for c in ['name', 'url', 'test_type_codes', 'duration_minutes', 'sim_score'] if c in candidates.columns]
    return candidates[cols].reset_index(drop=True)


print(f'--- Final Evaluation (alpha={best_alpha}) ---\n')
recalls_final = []
for _, row in train_grouped.iterrows():
    query = row['Query']
    actual_urls = row['Assessment_url']

    results = recommend_final(query, top_k=10)
    predicted_urls = results['url'].tolist()

    r = recall_at_k_v2(predicted_urls, actual_urls)
    recalls_final.append(r)
    print(f'Query: {query[:80]}...')
    print(f'  Relevant: {len(actual_urls)} | Recall@10: {r:.3f}')
    print()

mean_recall_final = np.mean(recalls_final)
print(f'=========================================')
print(f'  Baseline Mean Recall@10:  {mean_recall:.4f}')
print(f'  V2 Mean Recall@10:       {mean_recall_v2:.4f}')
print(f'  Final Mean Recall@10:    {mean_recall_final:.4f}')
print(f'=========================================')

--- Final Evaluation (alpha=0.7) ---

Query: Based on the JD below recommend me assessment for the Consultant position in my ...
  Relevant: 5 | Recall@10: 0.000

Query: Content Writer required, expert in English and SEO....
  Relevant: 5 | Recall@10: 0.600

Query: Find me 1 hour long assesment for the below job at SHL
Job Description

 Join a ...
  Relevant: 9 | Recall@10: 0.000

Query: I am hiring for Java developers who can also collaborate effectively with my bus...
  Relevant: 5 | Recall@10: 0.600

Query: I am looking for a COO for my company in China and I want to see if they are cul...
  Relevant: 6 | Recall@10: 0.000

Query: I want to hire a Senior Data Analyst with 5 years of experience and expertise in...
  Relevant: 10 | Recall@10: 0.100

Query: I want to hire new graduates for a sales role in my company, the budget is for a...
  Relevant: 9 | Recall@10: 0.333

Query: ICICI Bank Assistant Admin, Experience required 0-2 years, test should be 30-40 ...
  Relevant: 6 | Recall@1

In [32]:
# --- Final predictions on test set ---

predictions_final = []

for i, row in test_df.iterrows():
    query = row['Query']
    print(f'[{i+1}/{len(test_df)}] Query: {query[:80]}...')

    results = recommend_final(query, top_k=10)
    for _, r in results.iterrows():
        predictions_final.append({'Query': query, 'Assessment_url': r['url']})
    print(f'  -> {len(results)} recommendations\n')

pred_final_df = pd.DataFrame(predictions_final)
pred_final_df.to_csv('predictions.csv', index=False)
print(f'Final predictions saved to predictions.csv ({len(pred_final_df)} rows)')
pred_final_df.head(20)

[1/9] Query: Looking to hire mid-level professionals who are proficient in Python, SQL and Ja...
  -> 10 recommendations

[2/9] Query: Job Description

 Join a community that is shaping the future of work! 

 SHL, P...
  -> 10 recommendations

[3/9] Query: I am hiring for an analyst and wants applications to screen using Cognitive and ...
  -> 10 recommendations

[4/9] Query: I have a JD Job Description

 People Science. People Answers  !

Do you love con...
  -> 10 recommendations

[5/9] Query: I am new looking for new graduates in my sales team, suggest an 30 min long asse...
  -> 10 recommendations

[6/9] Query: For Marketing - Content Writer Position

Department: Marketing

Location: Gurugr...
  -> 10 recommendations

[7/9] Query: I want to hire a product manager with 3-4 years of work experience and expertise...
  -> 10 recommendations

[8/9] Query: Suggest me an assessment for the JD below Job Description

 Find purpose in each...
  -> 10 recommendations

[9/9] Query: I want to h

,Query,Assessment_url
0,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
1,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
2,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
3,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
4,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
5,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
6,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
7,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
8,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
9,Looking to hire mid-level professionals who ar...,https://www.shl.com/products/product-catalog/v...
